<!--
    Copyright (C) 2026 The AiQuokkaWannabe Authors <https://github.com/ai-quokka-wannabe/iso-lab>

    This program is free software: you can redistribute it and/or modify it under the terms of
    the GNU General Public License as published by the Free Software Foundation, either version
    3 of the License, or (at your option) any later version.

    This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY;
    without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
    See the GNU General Public License for more details.

    You should have received a copy of the GNU General Public License along with this program.
    If not, see <https://www.gnu.org/licenses/>.
-->

# The Benchmark's Power Curves: the Figures of Item 2 (e4)

This notebook draws, and decides nothing. It reads the table `power-curves` wrote
(`docs/benchmark/power-curves-2026-11-08.tsv`; the journal of 2026-11-08, § How Large an Alteration the Tests Can See)
and writes two SVG figures to `docs/figures/`, deterministically: `svg.hashsalt` is fixed and the date is left out of the
metadata, so an unchanged table redraws to the same bytes.

Each row of the table is one cell of the study: a sample size of block 2 against all of block 1, a measure, a direction
and a size in block 1's standard deviations, with four counts over the 100 replicates and each count's exact 95% interval.

In [ ]:
import csv
import os

import numpy as np
import matplotlib

matplotlib.use("svg")
import matplotlib.pyplot as plt

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

TABLE = "docs/benchmark/power-curves-2026-11-08.tsv"
with open(TABLE, encoding="utf-8") as handle:
    text = handle.read().splitlines()
comments = [line for line in text if line.startswith("#")]
A_ROWS = int(comments[0].split("(")[1].split(" ")[0])  # "# power-curves: a = ... (179 measured), ..."
rows = list(csv.DictReader([line for line in text if not line.startswith("#")], delimiter="\t"))
cells = [r for r in rows if r["measure"] != "none"]
nulls = [r for r in rows if r["measure"] == "none"]
MEASURES = list(dict.fromkeys(r["measure"] for r in cells))
SIZES = sorted({float(r["size_sd"]) for r in cells})
SAMPLES = sorted({int(r["sample_size"]) for r in cells})
REPLICATES = int(cells[0]["replicates"])
print(len(cells), "cells;", len(MEASURES), "measures; sizes", SIZES, "; sample sizes", SAMPLES, ";", REPLICATES, "replicates")
for r in nulls:
    print("null at n =", r["sample_size"], ": any flagged", r["flagged"], "(9999 shuffles:", r["flagged_more"] + ")", "; told apart", r["told_apart"])

## The curves

Each panel is one test and one direction; the x axis is the sample size of block 2's side, and each line is one
alteration size, the **median share over the 27 measures**, with the band from the lowest to the highest measure. The
intervals of each cell are in the table and in the heat map below, not here: a band over measures and an interval over
replicates would not be told apart on one axis.

In [ ]:
plt.rcParams["svg.hashsalt"] = "benchmark-power"
plt.rcParams["svg.fonttype"] = "none"  # text as text, not glyph outlines: the heat map carries 864 labels
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]  # one font name per label keeps the SVG small
BLUES = {0.25: "#9ecae1", 0.5: "#4292c6", 1.0: "#08519c", 2.0: "#08306b"}
TESTS = [("flagged_share", "per-measure comparison, 999 shuffles"), ("flagged_more_share", "per-measure comparison, 9999 shuffles"),
         ("told_apart_share", "classifier")]


def share(test, n, size, direction):
    return np.array([float(r[test]) for r in cells
                     if int(r["sample_size"]) == n and float(r["size_sd"]) == size and r["direction"] == direction])


fig, axes = plt.subplots(3, 2, figsize=(8.5, 8.5), sharex=True, sharey=True)
for row, (test, title) in enumerate(TESTS):
    for col, direction in enumerate(["up", "down"]):
        ax = axes[row, col]
        for size in SIZES:
            values = np.array([share(test, n, size, direction) for n in SAMPLES])
            ax.plot(SAMPLES, np.median(values, axis=1), "o-", color=BLUES[size], ms=4, lw=2, label=f"{size:g} sd")
            ax.fill_between(SAMPLES, values.min(axis=1), values.max(axis=1), color=BLUES[size], alpha=0.15, lw=0)
        ax.axhline(0.8, color="#8b949e", lw=0.8, ls=":")
        ax.set_title(f"{title}, {direction}", fontsize=9)
        ax.set_xscale("log")
        ax.set_xticks(SAMPLES)
        ax.set_xticklabels([str(n) for n in SAMPLES])
        ax.minorticks_off()
        ax.set_ylim(-0.02, 1.02)
        ax.grid(axis="y", color="#e6e6e6", lw=0.6)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
for ax in axes[-1]:
    ax.set_xlabel(f"worms on block 2's side (block 1's {A_ROWS} on the other)")
for ax in axes[:, 0]:
    ax.set_ylabel("share of 100 replicates seen")
axes[0, 0].legend(frameon=False, fontsize=8, title="shift, in block 1's sd", title_fontsize=8, loc="upper left")
fig.suptitle("Median over the 27 measures; the band spans the measures", fontsize=9)
fig.tight_layout()
fig.savefig("docs/figures/benchmark_power_curves.svg", metadata={"Date": None})
plt.close(fig)
print("written docs/figures/benchmark_power_curves.svg")

## The heat map

Every cell of the study at two sample sizes, 50 and all of block 2, for the per-measure comparison at its default
999 shuffles and for the classifier: one row per measure, one column per size and direction, the share seen written in
each cell. The intervals are in the table; at 100 replicates an exact 95% interval is at most about 10 points either
side of the share, and under 4 points at 0 or 1.

In [ ]:
SHOWN = [n for n in (50, max(SAMPLES)) if n in SAMPLES]
COLUMNS = [(direction, size) for direction in ("down", "up") for size in (sorted(SIZES, reverse=True) if direction == "down" else SIZES)]
PANELS = [("flagged_share", "per-measure comparison, 999 shuffles"), ("told_apart_share", "classifier")]
fig, axes = plt.subplots(len(SHOWN), 2, figsize=(9, 4.4 * len(SHOWN)), squeeze=False)
for row, n in enumerate(SHOWN):
    for col, (test, title) in enumerate(PANELS):
        ax = axes[row, col]
        grid = np.zeros((len(MEASURES), len(COLUMNS)))
        for r in cells:
            if int(r["sample_size"]) != n:
                continue
            i = MEASURES.index(r["measure"])
            j = COLUMNS.index((r["direction"], float(r["size_sd"])))
            grid[i, j] = float(r[test])
        ax.imshow(grid, cmap="Blues", vmin=0, vmax=1, aspect="auto")
        for i in range(len(MEASURES)):
            for j in range(len(COLUMNS)):
                ink = "white" if grid[i, j] > 0.6 else "#24292f"
                ax.text(j, i, f"{grid[i, j]:.2f}", ha="center", va="center", fontsize=6, color=ink)
        ax.set_xticks(range(len(COLUMNS)))
        ax.set_xticklabels([f"{'-' if d == 'down' else '+'}{s:g}" for d, s in COLUMNS], fontsize=7)
        ax.set_yticks(range(len(MEASURES)))
        ax.set_yticklabels(MEASURES if col == 0 else [], fontsize=6)
        ax.set_title(f"{title}; n = {n} against {A_ROWS}", fontsize=9)
        ax.tick_params(length=0)
        for side in ax.spines.values():
            side.set_visible(False)
for ax in axes[-1]:
    ax.set_xlabel("shift, in block 1's sd (down, then up)", fontsize=8)
fig.tight_layout()
fig.savefig("docs/figures/benchmark_power_heatmap.svg", metadata={"Date": None})
plt.close(fig)
print("written docs/figures/benchmark_power_heatmap.svg")

## The summary the plan asked for

Per measure, direction, sample size and test: the smallest of the four sizes at which the share seen is at least 0.8,
with that cell's interval, or *none*. Then the four expectations of the plan, checked as written.

In [ ]:
def cell(n, measure, direction, size):
    for r in cells:
        if int(r["sample_size"]) == n and r["measure"] == measure and r["direction"] == direction and float(r["size_sd"]) == size:
            return r
    raise KeyError((n, measure, direction, size))


def resolution(n, measure, direction, test):
    for size in SIZES:
        r = cell(n, measure, direction, size)
        if float(r[test + "_share"]) >= 0.8:
            return size, float(r[test + "_lo"]), float(r[test + "_hi"])
    return None


print("resolution: smallest size (sd of block 1) seen in at least 80 of 100 replicates; '-' is none up to 2 sd")
header = "measure".ljust(30) + "".join(f"  n={n}: 999 / 9999 / clf (up | down)".ljust(44) for n in SAMPLES)
print(header)
counts = {}
for measure in MEASURES:
    line = measure.ljust(30)
    for n in SAMPLES:
        parts = []
        for test in ("flagged", "flagged_more", "told_apart"):
            found = [resolution(n, measure, d, test) for d in ("up", "down")]
            parts.append("/".join("-" if f is None else f"{f[0]:g}" for f in found))
            for d, f in zip(("up", "down"), found):
                counts.setdefault((n, test, d), []).append(f[0] if f else float("inf"))
        line += ("  " + " ".join(parts)).ljust(44)
    print(line)
print()
print("how many of the 27 measures each test resolves at each size or better (up | down):")
for n in SAMPLES:
    for test in ("flagged", "flagged_more", "told_apart"):
        up, down = counts[(n, test, "up")], counts[(n, test, "down")]
        print(f"n={n:3d} {test:13s}", "  ".join(f"<= {s:g} sd: {sum(v <= s for v in up):2d} | {sum(v <= s for v in down):2d}" for s in SIZES))

In [ ]:
FULL = max(SAMPLES)
POSTURE = [m for m in MEASURES if m.startswith("arc") or m.startswith("eigen")]
e1 = cell(FULL, "eigen1_amplitude_sd[rad]", "up", 1.0)
print("1. eigen1 up 1 sd at n =", FULL, ": per-measure", e1["flagged_share"], f"({e1['flagged_lo']}-{e1['flagged_hi']}),",
      "classifier", e1["told_apart_share"], f"({e1['told_apart_lo']}-{e1['told_apart_hi']});",
      "refuted" if min(float(e1["flagged_share"]), float(e1["told_apart_share"])) < 0.9 else "as expected")
for test in ("flagged_share", "told_apart_share"):
    down_ge_up = sum(float(cell(FULL, m, "down", 1.0)[test]) >= float(cell(FULL, m, "up", 1.0)[test]) for m in POSTURE)
    print(f"2. {test}: downward share at least the upward one in {down_ge_up} of {len(POSTURE)} posture measures at 1 sd, n = {FULL};",
          "refuted" if 2 * down_ge_up >= len(POSTURE) else "as expected")
for r in nulls:
    bad = int(r["told_apart"]) > 10 or int(r["flagged"]) > 10
    print(f"3. null at n = {r['sample_size']}: told apart {r['told_apart']} ({r['told_apart_lo']}-{r['told_apart_hi']}), any flagged {r['flagged']}"
          f" ({r['flagged_lo']}-{r['flagged_hi']}), any flagged at 9999 {r['flagged_more']};", "LEVEL NOT HELD" if bad else "level held")
rises = 0
for m in MEASURES:
    for d in ("up", "down"):
        shares = [float(cell(n, m, d, 1.0)["told_apart_share"]) for n in SAMPLES]
        rises += all(a <= b for a, b in zip(shares, shares[1:]))
print(f"4a. classifier share at 1 sd never falls with the sample size in {rises} of {2 * len(MEASURES)} measure-directions")
worst = max((float(r[t]), r["measure"], r["direction"], r["sample_size"], t) for r in cells if float(r["size_sd"]) == 0.25 for t in ("flagged_share", "told_apart_share"))
print("4b. largest share at 0.25 sd:", worst, "; refuted" if worst[0] > 0.2 else "; as expected")